## Vicsek Model

The particle velocity evolves as,
$$
    \boldsymbol{v}_i = v_0 (\cos{\theta_i}, \sin{\theta_i}). \\
    \theta_{i+1} = \langle\theta_i\rangle_{\text{nearby particles}} + \eta(t)\Delta t
$$
The position changes as,
$$
    \boldsymbol{x}_{i+1} = \boldsymbol{x}_i + \boldsymbol{v}_i \Delta t.
$$
Here $i$ labels the time step. The average is the direction of the nearby particles' mean velocity, and $\eta(t)$ is a fresh random angular rate at each step.


In [ ]:
import numpy as np


In [ ]:
# Initialization (angles in radians).
N = 100          # Number of particles.
L = 10.0         # Box side length.
v_0 = 0.1        # Fixed speed.
R = 1.0          # Neighbor radius.
eta = 0.3        # Random angular-rate width; 0 means no noise.
dt = 1.0
n_steps = 200
seed = 42        # Same seed gives the same simulation.


In [ ]:
# Direction update: align with neighbors, then add random noise.
def update_theta(x, theta, R, L, eta, rng, dt=1.0):
    displacement = x[:, None, :] - x[None, :, :]
    displacement -= L * np.round(displacement / L)  # Periodic distances.
    neighbors = np.sum(displacement**2, axis=2) <= R**2  # Includes self.

    sum_cos = neighbors @ np.cos(theta)
    sum_sin = neighbors @ np.sin(theta)
    average = np.arctan2(sum_sin, sum_cos)
    # If directions cancel exactly, keep the particle's previous heading.
    average = np.where(np.hypot(sum_cos, sum_sin) > 1e-12, average, theta)
    noise = rng.uniform(-eta / 2, eta / 2, size=len(theta)) * dt
    return (average + noise) % (2 * np.pi)


In [ ]:
# Velocity update: only direction changes; speed stays fixed.
def update_velocity(theta, v_0):
    return v_0 * np.column_stack((np.cos(theta), np.sin(theta)))


In [ ]:
# Evolve and store every step.
def vicsek_model(N, L, v_0, R, eta, *, dt=1.0, n_steps=200, seed=None):
    """Return x, v, theta, t for particles in a 2D periodic box.

    x and v: (n_steps + 1, N, 2); theta: (n_steps + 1, N).
    eta is the random angular-rate width; angular noise is multiplied by dt.
    """
    if N < 1 or n_steps < 0 or L <= 0 or dt <= 0:
        raise ValueError("N and L/dt must be positive; n_steps must be nonnegative.")
    if v_0 < 0 or R < 0 or eta < 0:
        raise ValueError("v_0, R, and eta must be nonnegative.")

    rng = np.random.default_rng(seed)
    t = np.arange(n_steps + 1) * dt
    x = np.zeros((len(t), N, 2))
    v = np.zeros_like(x)
    theta = np.zeros((len(t), N))
    x[0] = rng.uniform(0, L, size=(N, 2))
    theta[0] = rng.uniform(0, 2 * np.pi, size=N)
    v[0] = update_velocity(theta[0], v_0)

    for n in range(n_steps):
        theta[n + 1] = update_theta(x[n], theta[n], R, L, eta, rng, dt)
        v[n + 1] = update_velocity(theta[n + 1], v_0)
        x[n + 1] = (x[n] + dt * v[n]) % L

    return x, v, theta, t


In [ ]:
x, v, theta, t = vicsek_model(
    N, L, v_0, R, eta, dt=dt, n_steps=n_steps, seed=seed
)

# x[n, i], v[n, i], theta[n, i]: particle i at time t[n].
x.shape, v.shape, theta.shape, t.shape


In [ ]:
# Visualization: arrows show direction; color encodes theta.
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML, Video, display
from pathlib import Path

alignment = np.hypot(np.mean(np.cos(theta), axis=1),
                      np.mean(np.sin(theta), axis=1))
with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10,
                     "animation.embed_limit": 80}):
    fig = plt.figure(figsize=(9, 6.5), dpi=85, facecolor="#080e1c")
    ax = fig.add_axes([0.08, 0.13, 0.58, 0.67], facecolor="#0c1629")
    chart = fig.add_axes([0.74, 0.36, 0.21, 0.23], facecolor="#0c1629")
    fig.text(0.08, 0.94, "VICSEK / COLLECTIVE MOTION", color="#6ce6ef",
             fontsize=10, weight="bold")
    fig.text(0.08, 0.875, "Simple rules. Moving together.", color="#edf5ff",
             fontsize=23, weight="bold")
    fig.text(0.08, 0.045, "PERIODIC BOX   /   FIXED SPEED   /   LOCAL ALIGNMENT + NOISE",
             color="#879bb5", fontsize=9)
    ax.set(xlim=(0, L), ylim=(0, L), xlabel="x", ylabel="y")
    ax.set_aspect("equal")
    ax.grid(color="#28405a", alpha=0.35, linewidth=0.5)
    for panel in (ax, chart):
        panel.tick_params(colors="#879bb5", labelsize=8)
        panel.xaxis.label.set_color("#b7c9df")
        panel.yaxis.label.set_color("#b7c9df")
        for spine in panel.spines.values():
            spine.set_color("#28405a")
    # Arrow lengths indicate heading, even if v_0 is zero.
    arrow_length = 0.025 * L
    arrows = ax.quiver(x[0, :, 0], x[0, :, 1],
                       arrow_length * np.cos(theta[0]), arrow_length * np.sin(theta[0]),
                       theta[0] % (2 * np.pi), cmap="hsv", clim=(0, 2 * np.pi),
                       angles="xy", scale_units="xy", scale=1, pivot="mid", width=0.005)
    chart.set(xlim=(0, t[-1] if t[-1] > 0 else 1), ylim=(0, 1.03), xlabel="time")
    chart.set_title("COLLECTIVE ALIGNMENT", color="#b7c9df", fontsize=10)
    history, = chart.plot([], [], color="#6ce6ef", lw=1.8)
    fig.text(0.74, 0.22, "1 = aligned\n0 = cancelling directions\nColors show heading",
             color="#879bb5", fontsize=9, linespacing=1.8, va="top")
    readout = fig.text(0.74, 0.78, "", color="#d3e5f8", family="monospace", va="top")
    fps = 24
    frames = np.unique(np.linspace(0, len(t) - 1, min(len(t), 289)).astype(int))

    def update(i):
        arrows.set_offsets(x[i])
        arrows.set_UVC(arrow_length * np.cos(theta[i]), arrow_length * np.sin(theta[i]),
                       theta[i] % (2 * np.pi))
        history.set_data(t[:i + 1], alignment[:i + 1])
        readout.set_text(f"t       {t[i]:6.2f}\nN       {N:6d}\neta     {eta:6.2f}\nalign   {alignment[i]:6.3f}")
        return arrows, history, readout

    animation = FuncAnimation(fig, update, frames=frames, interval=1000 / fps,
                               blit=False, repeat=True)
    player = animation.to_jshtml(default_mode="loop")
    plt.close(fig)

display(HTML(player))


In [ ]:
# Save the live HTML player and a fixed MP4 of the notebook simulation.
import imageio_ffmpeg
from vicsek_player import make_vicsek_player

controlled_player = make_vicsek_player(N, L, v_0, R, eta, dt, n_steps, seed)
Path("vicsek.html").write_text(controlled_player, encoding="utf-8")
with plt.rc_context({"animation.ffmpeg_path": imageio_ffmpeg.get_ffmpeg_exe()}):
    animation.save("vicsek.mp4", writer=FFMpegWriter(fps=fps), dpi=120)

display(Video("vicsek.mp4", embed=True))


### KDE velocity field and order–disorder scan (2-D)

The KDE cell adds animated periodic Gaussian averaging of the recorded velocities, with an inline player and `vicsek_kde.mp4`. Width, grid, trailing time window and density floor are editable. The separate `vicsek.html` player has matching live KDE controls.

The following scan varies number density $\rho=N/L^2$ and angular noise width $\eta\Delta t$, preserving the original solver and angular-rate convention. Measure
$$\Phi(t)=\left|\frac1N\sum_i e^{i\theta_i(t)}\right|,\qquad
\chi=N\left(\langle\Phi^2\rangle_t-\langle\Phi\rangle_t^2\right).$$
$\Phi$ near one indicates collective alignment; small values indicate orientational disorder, not mathematical chaos or absence of density clustering. Finite random flocks have nonzero $\Phi$ (of order $N^{-1/2}$). The maps show late-time averages across seeds and per-run temporal fluctuations; no arbitrary phase threshold is imposed. Peaks in $\chi$ are diagnostic, not an established transition boundary. Check longer runs, more seeds and larger systems for convergence.

[Original Vicsek order parameter](https://doi.org/10.1103/PhysRevLett.75.1226). Existing 2-D and 3-D model cells remain unchanged.


In [ ]:
# KDE average velocity field from the recorded particle motion.
# u(r) = sum K_h(r-x_i(t)) v_i(t) / sum K_h(r-x_i(t)).
# Average vectors first; |u| is different from the average particle speed.
import matplotlib.pyplot as plt


def periodic_kde_velocity(positions, velocities, box_length, bandwidth, *,
                          grid_size=32, min_density=0.0):
    """Return X, Y, mean velocity and number density on a periodic 2-D grid.

    Inputs: (frames, particles, 2), or a single (particles, 2) snapshot.
    Frames have equal weight. Wrapped Gaussian images enforce periodicity;
    image tails beyond at least eight bandwidths are neglected.
    Density is mean particle count per area, not a probability density.
    """
    positions = np.asarray(positions, dtype=float)
    velocities = np.asarray(velocities, dtype=float)
    if positions.ndim == 2:
        positions, velocities = positions[None], velocities[None]
    if (positions.shape != velocities.shape or positions.ndim != 3
            or positions.shape[-1] != 2 or min(positions.shape[:2]) < 1):
        raise ValueError("Use matching nonempty (frames, particles, 2) arrays.")
    if not (np.isfinite(positions).all() and np.isfinite(velocities).all()):
        raise ValueError("Positions and velocities must be finite.")
    if (not np.isfinite([box_length, bandwidth, min_density]).all()
            or box_length <= 0 or bandwidth <= 0 or min_density < 0):
        raise ValueError("L and bandwidth must be positive; density floor nonnegative.")
    if not isinstance(grid_size, (int, np.integer)) or grid_size < 2:
        raise ValueError("grid_size must be an integer of at least 2.")
    coordinates = np.arange(grid_size) * box_length / grid_size
    X, Y = np.meshgrid(coordinates, coordinates)
    queries = np.column_stack((X.ravel(), Y.ravel()))
    weight_sum = np.zeros(len(queries))
    momentum_sum = np.zeros((len(queries), 2))
    images = np.arange(-int(np.ceil(8 * bandwidth / box_length)),
                        int(np.ceil(8 * bandwidth / box_length)) + 1) * box_length
    # Process one frame at a time rather than allocate grid × frames × N.
    for frame_x, frame_v in zip(positions, velocities):
        displacement = queries[:, None, :] - frame_x[None, :, :]
        displacement -= box_length * np.round(displacement / box_length)
        kernel = np.ones(displacement.shape[:2])
        for component in range(2):
            axis_kernel = np.zeros_like(kernel)
            for shift in images:
                axis_kernel += np.exp(-0.5 * ((displacement[..., component] + shift)
                                              / bandwidth)**2)
            kernel *= axis_kernel
        weight_sum += kernel.sum(axis=1)
        momentum_sum += kernel @ frame_v
    density = weight_sum / (len(positions) * 2 * np.pi * bandwidth**2)
    supported = (density >= min_density) & (weight_sum > np.finfo(float).tiny)
    mean_velocity = np.full((len(queries), 2), np.nan)
    mean_velocity[supported] = momentum_sum[supported] / weight_sum[supported, None]
    return X, Y, mean_velocity.reshape(grid_size, grid_size, 2), density.reshape(X.shape)


# Animation controls: zero window gives an instantaneous spatial average.
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML, Video, display
import imageio_ffmpeg

kde_bandwidth = 0.06 * L
kde_grid_size = 32
kde_window = 0.0                  # Trailing averaging window in simulation time.
kde_max_window_samples = 7        # Equal-weight snapshots within that window.
kde_density_floor = 0.05 * N / L**2
kde_fps = 24
kde_frames = np.unique(np.linspace(0, len(t) - 1, min(len(t), 121)).astype(int))
kde_fields = []
for kde_index in kde_frames:
    kde_first = np.searchsorted(t, t[kde_index] - kde_window)
    kde_indices = np.unique(np.linspace(kde_first, kde_index,
        min(kde_max_window_samples, kde_index - kde_first + 1)).astype(int))
    kde_X, kde_Y, kde_u, kde_rho = periodic_kde_velocity(
        x[kde_indices], v[kde_indices], L, kde_bandwidth,
        grid_size=kde_grid_size, min_density=kde_density_floor
    )
    kde_fields.append(kde_u)
kde_fields = np.asarray(kde_fields)
kde_speeds = np.linalg.norm(kde_fields, axis=-1)
kde_finite_speeds = kde_speeds[np.isfinite(kde_speeds)]
kde_color_max = max(1e-12, float(kde_finite_speeds.max()) if kde_finite_speeds.size else 0.0)
with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10,
                     "animation.embed_limit": 80}):
    kde_fig, kde_ax = plt.subplots(figsize=(7.2, 6.2), layout="constrained",
                                   facecolor="#080e1c")
    kde_ax.set_facecolor("#0c1629")
    kde_edges = np.linspace(0, L, kde_grid_size + 1)
    kde_background = kde_ax.pcolormesh(
        kde_edges, kde_edges, np.pad(kde_speeds[0], ((0, 1), (0, 1)), mode="wrap"),
        shading="gouraud", cmap="magma", vmin=0, vmax=kde_color_max)
    kde_slice = (slice(None, None, 2),) * 2
    kde_arrows = kde_ax.quiver(kde_X[kde_slice], kde_Y[kde_slice],
        kde_fields[0][kde_slice][..., 0], kde_fields[0][kde_slice][..., 1],
        color="#d7f8ff", angles="xy", scale_units="xy", pivot="mid",
        scale=kde_color_max / (0.045 * L), width=0.004)
    kde_ax.set(xlim=(0, L), ylim=(0, L), xlabel="x", ylabel="y", aspect="equal")
    kde_title = kde_ax.set_title("", color="#edf5ff")
    kde_ax.tick_params(colors="#b7c9df")
    kde_ax.xaxis.label.set_color("#b7c9df")
    kde_ax.yaxis.label.set_color("#b7c9df")
    for spine in kde_ax.spines.values():
        spine.set_color("#28405a")
    kde_bar = kde_fig.colorbar(kde_background, ax=kde_ax, shrink=0.8)
    kde_bar.set_label("Magnitude of KDE mean velocity", color="#b7c9df")
    kde_bar.ax.tick_params(colors="#b7c9df")
    kde_fig.supxlabel("Periodic Gaussian KDE • trailing time window • low-density regions masked",
                     color="#879bb5", fontsize=9)

    def update_kde(frame):
        kde_background.set_array(np.pad(kde_speeds[frame], ((0, 1), (0, 1)), mode="wrap"))
        kde_arrows.set_UVC(kde_fields[frame][kde_slice][..., 0],
                           kde_fields[frame][kde_slice][..., 1])
        kde_title.set_text(f"KDE mean velocity | h={kde_bandwidth:g}\n"
                           f"t={t[kde_frames[frame]]:.3f} | window={kde_window:g}")
        return kde_background, kde_arrows, kde_title

    kde_animation = FuncAnimation(kde_fig, update_kde, frames=len(kde_frames),
                                  interval=1000 / kde_fps, blit=False, repeat=True)
    kde_player = kde_animation.to_jshtml(default_mode="loop")
    plt.close(kde_fig)
display(HTML(kde_player))

# Fixed video of this notebook's KDE run; HTML controls are saved by the earlier export cell.
with plt.rc_context({"animation.ffmpeg_path": imageio_ffmpeg.get_ffmpeg_exe()}):
    kde_animation.save("vicsek_kde.mp4", writer=FFMpegWriter(fps=kde_fps), dpi=120)
display(Video("vicsek_kde.mp4", embed=True))


In [ ]:
# Independent 2-D phase-scan settings; main simulation defaults stay unchanged.
vicsek_phase_N = 64
vicsek_phase_density = np.array([0.2, 0.5, 1.0, 2.0])
vicsek_phase_dt = dt
vicsek_phase_eta = np.linspace(0, 2 * np.pi / vicsek_phase_dt, 7)
vicsek_phase_steps = 600
vicsek_phase_seeds = (42, 43, 44)
vicsek_phase_burn_fraction = 0.5
vicsek_phase_v_0, vicsek_phase_R = v_0, R


In [ ]:
# Scan with the existing solver, storing finite-size order diagnostics per seed.
vicsek_phase_shape = (len(vicsek_phase_density), len(vicsek_phase_eta), len(vicsek_phase_seeds))
vicsek_phase_order = np.empty(vicsek_phase_shape)
vicsek_phase_chi = np.empty(vicsek_phase_shape)
vicsek_phase_early_order = np.empty(vicsek_phase_shape)
vicsek_phase_burn = int(vicsek_phase_steps * vicsek_phase_burn_fraction)
vicsek_phase_history = np.empty(vicsek_phase_shape + (vicsek_phase_steps + 1 - vicsek_phase_burn,))
vicsek_phase_done = 0
for row, density in enumerate(vicsek_phase_density):
    for col, noise_rate in enumerate(vicsek_phase_eta):
        for repeat, run_seed in enumerate(vicsek_phase_seeds):
            scan_x, scan_v, scan_theta, scan_t = vicsek_model(
                vicsek_phase_N, np.sqrt(vicsek_phase_N / density),
                vicsek_phase_v_0, vicsek_phase_R, noise_rate,
                dt=vicsek_phase_dt, n_steps=vicsek_phase_steps, seed=run_seed
            )
            scan_phi = np.abs(np.mean(np.exp(1j * scan_theta), axis=1))
            scan_late = scan_phi[vicsek_phase_burn:]
            vicsek_phase_order[row, col, repeat] = scan_late.mean()
            vicsek_phase_chi[row, col, repeat] = vicsek_phase_N * scan_late.var()
            vicsek_phase_early_order[row, col, repeat] = scan_phi[:vicsek_phase_burn].mean()
            vicsek_phase_history[row, col, repeat] = scan_late
            del scan_x, scan_v, scan_theta, scan_t
            vicsek_phase_done += 1
        print(f"{vicsek_phase_done}/{np.prod(vicsek_phase_shape)} runs: "
              f"rho={density:g}, angular noise={noise_rate*vicsek_phase_dt:.2f}, "
              f"mean alignment={vicsek_phase_order[row, col].mean():.3f}", flush=True)


In [ ]:
# Continuous measured maps; no invented boundary between ordered/disordered states.
import matplotlib.pyplot as plt
from pathlib import Path

vicsek_phase_noise = vicsek_phase_eta * vicsek_phase_dt
vicsek_phase_x_edges = np.r_[0, (vicsek_phase_noise[:-1] + vicsek_phase_noise[1:]) / 2, 2*np.pi]
vicsek_phase_y_mid = np.sqrt(vicsek_phase_density[:-1] * vicsek_phase_density[1:])
vicsek_phase_y_edges = np.r_[vicsek_phase_density[0]**2 / vicsek_phase_y_mid[0],
    vicsek_phase_y_mid, vicsek_phase_density[-1]**2 / vicsek_phase_y_mid[-1]]
with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10}):
    vicsek_phase_fig, vicsek_phase_axes = plt.subplots(1, 2, figsize=(11, 5.2),
        layout="constrained", facecolor="#080e1c")
    vicsek_phase_fig.suptitle("Vicsek / finite-size order–disorder map", color="#edf5ff", fontsize=18)
    for panel, values, title, limits in zip(
        vicsek_phase_axes,
        [vicsek_phase_order.mean(axis=2), vicsek_phase_chi.mean(axis=2)],
        [r"Mean alignment $\langle\Phi\rangle$", r"Temporal fluctuations $N\,\mathrm{Var}(\Phi)$"],
        [{"vmin": 0, "vmax": 1}, {"vmin": 0}]
    ):
        mesh = panel.pcolormesh(vicsek_phase_x_edges, vicsek_phase_y_edges,
                                values, cmap="viridis", shading="flat", **limits)
        panel.set(yscale="log", xlabel=r"Angular noise width $\eta\Delta t$",
                  ylabel=r"Number density $\rho=N/L^2$")
        panel.set_yticks(vicsek_phase_density, [f"{value:g}" for value in vicsek_phase_density])
        panel.set_xticks([0, np.pi, 2*np.pi], ["0", "π", "2π"])
        panel.minorticks_off()
        panel.set_title(title, color="#edf5ff")
        panel.tick_params(colors="#b7c9df")
        panel.xaxis.label.set_color("#b7c9df")
        panel.yaxis.label.set_color("#b7c9df")
        for spine in panel.spines.values(): spine.set_color("#28405a")
        bar = vicsek_phase_fig.colorbar(mesh, ax=panel, shrink=0.8)
        bar.ax.tick_params(colors="#b7c9df")
    vicsek_phase_fig.supxlabel(
        f"N={vicsek_phase_N} | {vicsek_phase_steps} steps | {len(vicsek_phase_seeds)} seeds | "
        "High alignment = ordered motion; low alignment = disordered motion",
        color="#879bb5", fontsize=9)
    vicsek_phase_fig.savefig("vicsek_phase_map.png", dpi=180, facecolor=vicsek_phase_fig.get_facecolor())
    plt.show()
np.savez_compressed("vicsek_phase_scan.npz", density=vicsek_phase_density,
    eta=vicsek_phase_eta, dt=vicsek_phase_dt, angular_noise=vicsek_phase_noise,
    mean_order=vicsek_phase_order, fluctuations=vicsek_phase_chi,
    early_order=vicsek_phase_early_order, late_order_history=vicsek_phase_history,
    N=vicsek_phase_N, n_steps=vicsek_phase_steps, seeds=vicsek_phase_seeds,
    burn_fraction=vicsek_phase_burn_fraction, v_0=vicsek_phase_v_0, R=vicsek_phase_R)
print("Saved vicsek_phase_map.png and vicsek_phase_scan.npz in", Path.cwd())


## Vicsek model in 3-D

Keep your two-angle velocity convention:
$$
\boldsymbol{v}_i=v_0(\cos\theta_i\sin\phi_i,\;\cos\theta_i\cos\phi_i,\;\sin\theta_i).
$$
Here $\theta$ is elevation and $\phi$ is measured from positive $y$ toward positive $x$. We still store both angle histories.

**Alignment:** sum the nearby particles' unit directions into $\boldsymbol S$ and normalize:
$$\bar{\boldsymbol u}_i=\frac{\boldsymbol S}{|\boldsymbol S|}.$$

**Random turning:** choose a new direction uniformly within a cone around $\bar{\boldsymbol u}_i$. Its maximum turning angle is
$$\alpha=\min(\pi,\eta\Delta t/2).$$
Generate the two independent random variables
$$c=\cos\beta\sim U[\cos\alpha,1],\qquad \psi\sim U[0,2\pi).$$
If $\boldsymbol e_1,\boldsymbol e_2$ are perpendicular unit vectors tangent to the mean heading, the new unit direction is
$$
\boldsymbol u_{i+1}=c\,\bar{\boldsymbol u}_i+
\sqrt{1-c^2}\left(\cos\psi\,\boldsymbol e_1+\sin\psi\,\boldsymbol e_2\right).
$$
Sampling **cosine** of the turning angle uniformly makes the noise uniform in solid angle within the cone, with no preferred direction around the mean heading. We do not add independent noise to the coordinate angles.

Recover your angles from that new direction:
$$
\theta_{i+1}=\operatorname{atan2}(u_z,\sqrt{u_x^2+u_y^2}),\qquad
\phi_{i+1}=\operatorname{atan2}(u_x,u_y)\pmod{2\pi}.
$$
Then use the same velocity formula above. Zero noise gives the mean heading; when $\alpha=\pi$, every direction on the sphere is equally likely.

Positions keep your convention:
$$\boldsymbol{x}_{i+1}=\boldsymbol{x}_i+\boldsymbol{v}_i\Delta t.$$
All particles update together from the previous state, inside a periodic cube. If the neighbors' directions cancel, use the old heading as the cone axis. At an exactly vertical new heading, keep the previous $\phi$, since the azimuth is undefined. Initial headings are uniform on the sphere.


In [ ]:
# Separate 3-D initialization; the 2-D parameters stay untouched.
N_3d = 150
L_3d = 10.0
v_0_3d = 0.1
R_3d = 2.0
eta_3d = 0.3
dt_3d = 1.0
n_steps_3d = 200
seed_3d = 42


In [ ]:
# Uniform random turning on a spherical cap, while retaining theta/phi state.
def add_direction_noise_3d(average, eta, rng, dt=1.0):
    reference = np.tile([0.0, 0.0, 1.0], (len(average), 1))
    reference[np.abs(average[:, 2]) > 0.9] = [1.0, 0.0, 0.0]
    e1 = np.cross(reference, average)
    e1 /= np.linalg.norm(e1, axis=1)[:, None]
    e2 = np.cross(average, e1)
    alpha = min(np.pi, eta * dt / 2)
    cos_beta = rng.uniform(np.cos(alpha), 1.0, size=len(average))
    sin_beta = np.sqrt(np.maximum(0, 1 - cos_beta**2))
    psi = rng.uniform(0, 2 * np.pi, size=len(average))
    updated = (cos_beta[:, None] * average
               + (sin_beta * np.cos(psi))[:, None] * e1
               + (sin_beta * np.sin(psi))[:, None] * e2)
    return updated / np.linalg.norm(updated, axis=1)[:, None]


def update_angles_3d(x, theta, phi, R, L, eta, rng, dt=1.0):
    direction = update_velocity_3d(theta, phi, 1.0)
    displacement = x[:, None, :] - x[None, :, :]
    displacement -= L * np.round(displacement / L)
    neighbors = np.sum(displacement**2, axis=2) <= R**2
    mean = neighbors @ direction
    lengths = np.linalg.norm(mean, axis=1)
    average = mean / np.where(lengths > 1e-12, lengths, 1)[:, None]
    average[lengths <= 1e-12] = direction[lengths <= 1e-12]
    updated = add_direction_noise_3d(average, eta, rng, dt)
    horizontal = np.hypot(updated[:, 0], updated[:, 1])
    theta_new = np.arctan2(updated[:, 2], horizontal)
    phi_new = np.where(horizontal > 1e-12,
                       np.arctan2(updated[:, 0], updated[:, 1]), phi)
    return theta_new, phi_new % (2 * np.pi)


In [ ]:
# Your velocity convention: theta is elevation; phi starts from +y.
def update_velocity_3d(theta, phi, v_0):
    return v_0 * np.stack((np.cos(theta) * np.sin(phi),
                          np.cos(theta) * np.cos(phi),
                          np.sin(theta)), axis=-1)


In [ ]:
def vicsek_model_3d(N, L, v_0, R, eta, *, dt=1.0, n_steps=200, seed=None):
    """Return x, v, theta, phi, t, using uniform spherical-cap turns.

    x/v: (steps+1, N, 3); theta/phi: (steps+1, N).
    """
    if N < 1 or n_steps < 0 or L <= 0 or dt <= 0:
        raise ValueError("N, L, dt must be positive; steps must be nonnegative.")
    if v_0 < 0 or R < 0 or eta < 0:
        raise ValueError("v_0, R and eta must be nonnegative.")
    rng = np.random.default_rng(seed)
    t = np.arange(n_steps + 1) * dt
    x = np.zeros((len(t), N, 3))
    v = np.zeros_like(x)
    theta = np.zeros((len(t), N))
    phi = np.zeros_like(theta)
    x[0] = rng.uniform(0, L, size=(N, 3))
    theta[0] = np.arcsin(rng.uniform(-1, 1, size=N))
    phi[0] = rng.uniform(0, 2 * np.pi, size=N)
    v[0] = update_velocity_3d(theta[0], phi[0], v_0)
    for i in range(n_steps):
        theta[i + 1], phi[i + 1] = update_angles_3d(
            x[i], theta[i], phi[i], R, L, eta, rng, dt)
        v[i + 1] = update_velocity_3d(theta[i + 1], phi[i + 1], v_0)
        x[i + 1] = (x[i] + dt * v[i]) % L
    return x, v, theta, phi, t


In [ ]:
x_3d, v_3d, theta_3d, phi_3d, t_3d = vicsek_model_3d(
    N_3d, L_3d, v_0_3d, R_3d, eta_3d,
    dt=dt_3d, n_steps=n_steps_3d, seed=seed_3d
)
# Derived unit vectors are used only for arrows and alignment measurements.
direction_3d = update_velocity_3d(theta_3d, phi_3d, 1.0)
x_3d.shape, v_3d.shape, theta_3d.shape, phi_3d.shape, t_3d.shape


In [ ]:
# A separate 3-D animation and alignment history.
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML, Video, display
from pathlib import Path
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from matplotlib.ticker import MaxNLocator


def arrow_segments_3d(position, direction, length):
    reference = np.tile([0.0, 0.0, 1.0], (len(position), 1))
    reference[np.abs(direction[:, 2]) > 0.9] = [1.0, 0.0, 0.0]
    side = np.cross(reference, direction)
    side /= np.linalg.norm(side, axis=1)[:, None]
    tip = position + 0.65 * length * direction
    tail = position - 0.35 * length * direction
    wing = tip - 0.35 * length * direction
    return np.concatenate([np.stack([tail, tip], axis=1),
                           np.stack([tip, wing + 0.18 * length * side], axis=1),
                           np.stack([tip, wing - 0.18 * length * side], axis=1)])


alignment_3d = np.linalg.norm(np.mean(direction_3d, axis=1), axis=1)
with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10,
                     "grid.color": "#28405a", "grid.linewidth": 0.5,
                     "animation.embed_limit": 100}):
    fig_3d = plt.figure(figsize=(10, 6.5), dpi=85, facecolor="#080e1c")
    ax_3d = fig_3d.add_axes([0.01, 0.12, 0.69, 0.69], projection="3d", facecolor="#080e1c")
    chart_3d = fig_3d.add_axes([0.75, 0.36, 0.2, 0.23], facecolor="#0c1629")
    fig_3d.text(0.07, 0.94, "VICSEK / THREE DIMENSIONS", color="#6ce6ef", fontsize=10, weight="bold")
    fig_3d.text(0.07, 0.875, "A flock in a periodic cube.", color="#edf5ff", fontsize=23, weight="bold")
    fig_3d.text(0.07, 0.045, "FIXED SPEED   /   LOCAL ALIGNMENT   /   UNIFORM CONE NOISE / THETA + PHI STATE", color="#879bb5", fontsize=9)
    ax_3d.set(xlim=(0, L_3d), ylim=(0, L_3d), zlim=(0, L_3d))
    ax_3d.set_box_aspect((1, 1, 1))
    ax_3d.set_proj_type("ortho")
    ax_3d.view_init(elev=22, azim=-55)
    for axis, name in zip((ax_3d.xaxis, ax_3d.yaxis, ax_3d.zaxis), ("x", "y", "z")):
        axis.set_pane_color((0.05, 0.09, 0.16, 1))
        axis.set_label_text(name)
        axis.label.set_color("#b7c9df")
        axis.line.set_color("#28405a")
        axis.set_major_locator(MaxNLocator(3))
    ax_3d.tick_params(colors="#879bb5", labelsize=8)
    hues_3d = phi_3d[0] % (2 * np.pi)
    colors_3d = plt.get_cmap("hsv")(hues_3d / (2 * np.pi))
    arrows_3d = Line3DCollection(
        arrow_segments_3d(x_3d[0], direction_3d[0], 0.035 * L_3d),
        colors=np.tile(colors_3d, (3, 1)), linewidths=1.3, alpha=0.9)
    ax_3d.add_collection3d(arrows_3d)
    chart_3d.set(xlim=(0, t_3d[-1] if t_3d[-1] > 0 else 1), ylim=(0, 1.03), xlabel="time")
    chart_3d.set_title("HEADING ALIGNMENT", color="#b7c9df", fontsize=10)
    chart_3d.tick_params(colors="#879bb5", labelsize=8)
    chart_3d.xaxis.label.set_color("#b7c9df")
    for spine in chart_3d.spines.values():
        spine.set_color("#28405a")
    history_3d, = chart_3d.plot([], [], color="#6ce6ef", lw=1.8)
    readout_3d = fig_3d.text(0.75, 0.78, "", color="#d3e5f8", family="monospace", va="top")
    fig_3d.text(0.75, 0.22, "1 = aligned\n0 = cancelling directions\nColor: phi heading",
                color="#879bb5", fontsize=9, linespacing=1.8, va="top")
    fps_3d = 24
    frames_3d = np.unique(np.linspace(0, len(t_3d) - 1, min(len(t_3d), 289)).astype(int))

    def update_3d(i):
        arrows_3d.set_segments(arrow_segments_3d(x_3d[i], direction_3d[i], 0.035 * L_3d))
        hues = phi_3d[i] % (2 * np.pi)
        arrows_3d.set_color(np.tile(plt.get_cmap("hsv")(hues / (2 * np.pi)), (3, 1)))
        history_3d.set_data(t_3d[:i + 1], alignment_3d[:i + 1])
        readout_3d.set_text(f"t       {t_3d[i]:6.2f}\nN       {N_3d:6d}\neta     {eta_3d:6.2f}\nalign   {alignment_3d[i]:6.3f}")
        return arrows_3d, history_3d, readout_3d

    animation_3d = FuncAnimation(fig_3d, update_3d, frames=frames_3d, interval=1000 / fps_3d, blit=False, repeat=True)
    player_3d = animation_3d.to_jshtml(default_mode="loop")
    plt.close(fig_3d)

display(HTML(player_3d))


In [ ]:
# Separate names preserve the 2-D HTML and video.
import imageio_ffmpeg
from vicsek_3d_player import make_vicsek_3d_player

controlled_player_3d = make_vicsek_3d_player(
    N_3d, L_3d, v_0_3d, R_3d, eta_3d, dt_3d, n_steps_3d, seed_3d
)
Path("vicsek_3d.html").write_text(controlled_player_3d, encoding="utf-8")
with plt.rc_context({"animation.ffmpeg_path": imageio_ffmpeg.get_ffmpeg_exe()}):
    animation_3d.save("vicsek_3d.mp4", writer=FFMpegWriter(fps=fps_3d), dpi=120)

display(Video("vicsek_3d.mp4", embed=True))
